# GENE CRISpy — M1 Genetics Foundations

Independent study aid mapped to selected public UIC College of Medicine Block 1 objectives. Not UIC-authored or endorsed; not a complete syllabus. Synthetic examples only. No clinical interpretation, patient risk scoring, variant annotation or CRISPR design.

Source: https://medicine.uic.edu/education/md/md-curriculum/phase-1-curriculum/

Reviewed 2026-11-14. Phase 1 includes M1 and M2; this notebook covers selected Block 1 connections only.


## 1. Load the core package
CPU is sufficient. Each run downloads the genetics branch into a fresh folder, preserving previous Colab files. No Firebase, API server, credentials or GPU is required. The printed Git commit identifies the loaded version.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

# Use a fresh folder each run; preserve all previous Colab files.
repo = Path(tempfile.mkdtemp(prefix="genecrispy-m1-", dir="/content")) / "repository"
result = subprocess.run([
    "git", "clone", "--branch", "genecrispy-m1-genetics", "--single-branch",
    "https://github.com/contrivancecompanychicago/replicatedgenecrispy2dpatientworld.git",
    str(repo)
], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError("Repository download failed:\n" + result.stderr)

for name in list(sys.modules):
    if name == "genecrispy" or name.startswith("genecrispy."):
        del sys.modules[name]
sys.path.insert(0, str(repo))

from genecrispy import GeneCrispyAgent, __version__
from genecrispy import get_m1_genetics_curriculum, get_genetics_lesson

agent = GeneCrispyAgent()
print("GENE CRISpy version:", __version__)
print("M1 genetics lessons:", len(get_m1_genetics_curriculum()["lessons"]))
print("Loaded from:", repo)
subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], check=True)


## 2. Read the learning map
The objective connections below are our study mapping, not additional official UIC objectives. B1.4 connects inheritance and genetic testing; B1.8 includes pharmacogenomics; B1.11 supports evidence appraisal; B1.13 includes ethical and structural context.


In [ ]:
curriculum = get_m1_genetics_curriculum()
print(curriculum["notice"])
print("Source:", curriculum["source_url"])
for lesson in curriculum["lessons"]:
    print(lesson["id"], "—", lesson["title"], "—", ", ".join(lesson["objective_ids"]))


## 3. Repeat your three DNA examples and add RNA
FASTA is a text format. GC percentage is a composition statistic, not a measure of disease risk. GCGCGC does not establish DNA versus RNA identity from letters alone.


In [ ]:
samples = ">sample_A\nACGTGC\n>sample_B\nATATAT\n>sample_C\nGCGCGC\n>sample_RNA\nACGU\n"
results = agent.analyze(samples, input_format="fasta")
for record in results["records"]:
    print(record["id"], record["molecule_type"], "length:", record["length"],
          "GC%:", record["gc_percent_of_canonical_bases"])
assert results["records"][3]["molecule_type"] == "RNA"


## 4. Learn and reflect
Read each explanation, write your own response, then compare with the model answer in the next cell. No learner responses are collected or sent anywhere.


In [ ]:
for lesson in curriculum["lessons"]:
    print("\n" + lesson["title"])
    print(lesson["explanation"])
    print("Reflect:", lesson["reflection"])


In [ ]:
for lesson in curriculum["lessons"]:
    print(lesson["id"] + ":", lesson["answer"])


## 5. Check the genetics learning path
These focused checks do not load the legacy PatientWorld/API modules. The pre-existing full test suite still has unrelated import blockers.


In [ ]:
subprocess.run([
    sys.executable, "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_m1_education.py", "-v"
], cwd=repo, check=True)


## 6. Download your results (optional)
Downloads a CSV and JSON bundle. Save a copy of the notebook to your Google Drive as well; runtime files are temporary.


In [ ]:
import csv
import json
import zipfile
from google.colab import files

csv_path = Path("/content/GENE_CRISpy_M1_results.csv")
fields = ["id", "molecule_type", "length", "gc_percent_of_canonical_bases", "ambiguous_symbol_count"]
with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results["records"])
json_path = Path("/content/GENE_CRISpy_M1_learning.json")
json_path.write_text(json.dumps({"curriculum": curriculum, "results": results}, indent=2))
bundle = Path("/content/GENE_CRISpy_M1_results.zip")
with zipfile.ZipFile(bundle, "w") as archive:
    archive.write(csv_path, csv_path.name)
    archive.write(json_path, json_path.name)
files.download(str(bundle))
